In [ ]:
import os
from pathlib import Path
from ultralytics import YOLO

PROJECT_ROOT = Path.cwd().parent  # notebooks/ → robomaster-cv/
DATA_DIR = PROJECT_ROOT / 'data/mergeRM'
DATASET_YAML = str(PROJECT_ROOT / 'configs/data.yaml')

os.listdir(DATA_DIR)

['data.yaml', 'images', 'labels', '.ipynb_checkpoints']

In [2]:
IMG_DIR = DATA_DIR / 'images'
LABEL_DIR = DATA_DIR / 'labels'

print(os.listdir(IMG_DIR))
print(os.listdir(LABEL_DIR))

['val', 'test', 'train']
['val', 'val.cache', 'train.cache', 'test', 'train']


In [3]:
for dataset in os.listdir(IMG_DIR):
    print(f'{dataset.capitalize()}:', len(os.listdir(IMG_DIR / dataset)), 'and', len(os.listdir(IMG_DIR / dataset)))

Val: 909 and 909
Test: 706 and 706
Train: 5805 and 5805


In [4]:
model = YOLO('yolo11s.pt')

hyp_search_space = {
    'lr0': (1e-5, 1e-1),
    'momentum': (0.6, 0.98),
    'weight_decay': (0.0, 0.001),
    'box': (5, 10),
    'mosaic': (0.0, 1.0)
}

!nvidia-smi

Fri Feb 13 14:01:32 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 570.207                Driver Version: 570.207        CUDA Version: 12.8     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA GeForce RTX 4090        On  |   00000000:01:00.0 Off |                  Off |
|  0%   37C    P8             18W /  450W |      97MiB /  24564MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [5]:
%%time


model.tune(
    data=DATASET_YAML,
    epochs=10,
    iterations=10,    # Keep this low for testing; increase for serious optimization
    imgsz=640,        # Standard image size for YOLO
    name="tune_run",  # Name of the folder where results are saved
    space=hyp_search_space
)

Tuner: Initialized Tuner instance with 'tune_dir=/home/jupyter-robomaster/jithendra/runs/detect/tune_run'
Tuner: 💡 Learn about tuning at https://docs.ultralytics.com/guides/hyperparameter-tuning
Tuner: Starting iteration 1/10 with hyperparameters: {'lr0': 0.01, 'momentum': 0.937, 'weight_decay': 0.0005, 'box': 7.5, 'mosaic': 1.0}
Ultralytics 8.4.14 🚀 Python-3.9.7 torch-2.6.0+cu118 CUDA:0 (NVIDIA GeForce RTX 4090, 24081MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=data/mergeRM/data.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=10, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7

In [5]:
!nvidia-smi

Fri Feb 13 14:01:40 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 570.207                Driver Version: 570.207        CUDA Version: 12.8     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA GeForce RTX 4090        On  |   00000000:01:00.0 Off |                  Off |
|  0%   37C    P8             18W /  450W |      97MiB /  24564MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [6]:
%%time

model.train(
    data = DATASET_YAML,
    epochs = 100,
    weight_decay = 0.001,
    box = 6.22111,
    imgsz = 640,
    name = 'final_model'
)

Ultralytics 8.4.14 🚀 Python-3.9.7 torch-2.6.0+cu118 CUDA:0 (NVIDIA GeForce RTX 4090, 24081MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=6.22111, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=data/mergeRM/data.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=100, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11s.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=final_model2, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap_mask=True, patience

ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([0])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x731d073f6730>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0.033033,    0.034034,    0.035035,    0.036036,    0.037037,    0.038038,    0.039039,     0.04004,    0.041041,    0.042042,    0.043043,    0.044044,    0.045045,    0.046046,    0.047047,
          0.048048, 

## Inference

In [7]:
# PATH SETUP: Update these paths to match your actual folder structure!
# Point this to the 'best.pt' file created during your training step above.
path_to_best_model = "/home/jupyter-robomaster/jithendra/runs/detect/final_model2/weights/best.pt"
test_images_path = "/home/jupyter-robomaster/jithendra/data/mergeRM/images/test"
output_path = "/home/jupyter-robomaster/jithendra/inference_results"

# Load your custom trained model
best_model = YOLO(path_to_best_model)

# Validate metrics 
metrics = best_model.val()

Ultralytics 8.4.14 🚀 Python-3.9.7 torch-2.6.0+cu118 CUDA:0 (NVIDIA GeForce RTX 4090, 24081MiB)
YOLO11s summary (fused): 101 layers, 9,413,187 parameters, 0 gradients, 21.3 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 2966.3±1093.1 MB/s, size: 45.3 KB)
val: Scanning /home/jupyter-robomaster/jithendra/data/mergeRM/labels/val.cache... 742 images, 166 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 908/908 211.6Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 57/57 18.1it/s 3.1s0.1s
                   all        908       1211      0.911      0.913      0.956       0.69
Speed: 0.6ms preprocess, 1.3ms inference, 0.0ms loss, 0.5ms postprocess per image
Results saved to /home/jupyter-robomaster/jithendra/runs/detect/val


In [8]:
metrics = best_model.val(split='test')

Ultralytics 8.4.14 🚀 Python-3.9.7 torch-2.6.0+cu118 CUDA:0 (NVIDIA GeForce RTX 4090, 24081MiB)
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 55.6±67.7 MB/s, size: 37.0 KB)
val: Scanning /home/jupyter-robomaster/jithendra/data/mergeRM/labels/test... 573 images, 132 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 705/705 4.5Kit/s 0.2s0.2s
val: New cache created: /home/jupyter-robomaster/jithendra/data/mergeRM/labels/test.cache
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 45/45 16.9it/s 2.7s0.1s
                   all        705        927      0.867      0.892      0.922      0.635
Speed: 0.7ms preprocess, 1.2ms inference, 0.0ms loss, 0.3ms postprocess per image
Results saved to /home/jupyter-robomaster/jithendra/runs/detect/val2
